# 15 — Freeze a matched-backbone direct-image ablation on Biowulf

Run on a CPU allocation. No generation, training, or old-result edits.
This addresses the unequal-input concern without treating model families as exchangeable.
Same backbone, same retrieved reports, same neutral instructions, with/without direct query pixels.
Single-pass is primary. B/C/D are secondary whole-pipeline ablations; MedGemma verifier pixels
also change, while Phi's verifier remains text-only. Retrieval is image-conditioned in BOTH arms.

Start with `RUN_NAME = "matched_input_smoke_v1"` in every notebook; then freeze a separate
`matched_input_v1` production run. Never promote smoke outputs to manuscript results.
All original Notebooks 00–14 and results remain unchanged.

In [ ]:
from pathlib import Path
import os, sys, json

candidate = Path(os.environ.get("JAMIA_RERUN_DIR", Path.cwd())).resolve()
if not (candidate / "rerun_config.json").is_file() and (candidate.parent / "rerun_config.json").is_file():
    candidate = candidate.parent
RERUN_DIR = candidate
if not (RERUN_DIR / "rerun_config.json").is_file():
    raise FileNotFoundError("Copy rerun_config.example.json to rerun_config.json, configure paths, and set JAMIA_RERUN_DIR to the repository root")
implementation = (RERUN_DIR / "src" / "rerun_code").resolve()
sys.path[:] = [entry for entry in sys.path if Path(entry or ".").resolve() != implementation]
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config
_, CONFIG = load_config(RERUN_DIR)
RESULTS_ROOT = Path(os.environ.get("JAMIA_OUTPUT_ROOT", CONFIG["output_root"]))
CONFIG["output_root"] = str(RESULTS_ROOT)
RUN_NAME = os.environ.get("JAMIA_ABLATION_RUN", "matched_input_v1")
ABLATION_ROOT = RESULTS_ROOT / "post_rerun" / "modality_ablation" / RUN_NAME
print("Code:", RERUN_DIR)
print("Existing inputs:", RESULTS_ROOT)
print("NEW output:", ABLATION_ROOT)


In [ ]:
from rerun_code.modality_ablation import freeze_experiment, CONDITIONS

# Adjust BEFORE freezing. Changing these later requires a new RUN_NAME.
MODEL_KEYS = ["medgemma_4b", "phi4_multimodal"]
BUNDLE_NAMES = ["mimic", "iuhn", "combined"]
STRATEGIES = list(CONDITIONS)  # use ["A_single_pass"] for the minimal primary experiment
SMOKE_N = 2 if "smoke" in RUN_NAME.lower() else None

# Known image/JSON prefixes in saved manifests are resolved from /vf/users/liangz2/openi
# to /data/liangz2/openi when available, then checked against the ORIGINAL content hashes.
for relative in ["manifests/leakage_audit.json", "bundles", "verifiers"]:
    path = RESULTS_ROOT / relative
    if not path.exists():
        raise FileNotFoundError(f"Required earlier-run input missing on Biowulf: {path}")
output, PLAN = freeze_experiment(
    RERUN_DIR, CONFIG, run_name=RUN_NAME, models=MODEL_KEYS, bundles=BUNDLE_NAMES,
    conditions=STRATEGIES, smoke_n=SMOKE_N, seed=20261029,
)
print(json.dumps({key: PLAN[key] for key in ["experiment_id", "models", "bundles",
                 "conditions", "query_counts", "expected_records", "smoke_n"]}, indent=2))
print("Upload/run Notebook 16 next; no new fine-tuning is required.")